<a href="https://colab.research.google.com/github/Shy-coder16/PROMPT-LOGIC/blob/main/CRIBA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install gradio nltk spacy python-docx pypdf --quiet

import os
import docx
import pypdf
import nltk
import spacy
from nltk.corpus import stopwords
from nltk.tokenize import sent_tokenize
from collections import Counter
import gradio as gr

# Download resources
nltk.download('punkt_tab', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)

try:
    nlp = spacy.load("en_core_web_sm")
except OSError:
    spacy.cli.download("en_core_web_sm")
    nlp = spacy.load("en_core_web_sm")

# Extract text supporting docx and pdf
def extract_text(file_path):
    if file_path.endswith('.pdf') or 'PDF' in file_path:
        reader = pypdf.PdfReader(file_path)
        text = ""
        for page in reader.pages:
            text += page.extract_text() or ""
        return text
    else:
        doc = docx.Document(file_path)
        return "\n".join([para.text for para in doc.paragraphs])

# Summarization
def summarize(text, num_sentences=3):
    words = nltk.word_tokenize(text.lower())
    stop_words = set(stopwords.words("english"))
    words = [word for word in words if word.isalnum() and word not in stop_words]
    word_freq = Counter(words)
    sentences = sent_tokenize(text)
    sentence_scores = {}
    for sent in sentences:
        for word in nltk.word_tokenize(sent.lower()):
            if word in word_freq:
                sentence_scores[sent] = sentence_scores.get(sent, 0) + word_freq[word]
    ranked_sentences = sorted(sentence_scores, key=sentence_scores.get, reverse=True)
    summary = " ".join(ranked_sentences[:num_sentences])
    return summary

# Skill extraction
def extract_skills(text):
    doc = nlp(text.lower())
    skills = []
    for chunk in doc.noun_chunks:
        if any(token.pos_ in ['NOUN', 'PROPN'] for token in chunk):
            skills.append(chunk.text.strip())
    common_skills = Counter(skills).most_common(5)
    return [skill for skill, _ in common_skills]

# Main Function updated for Gradio file handling
def process_resume(file):
    if file is None:
        return "No file uploaded.", "No skills found."
    text = extract_text(file.name)
    summary = summarize(text)
    skills = extract_skills(text)
    skills_formatted = "\n".join([f"- {s.capitalize()}" for s in skills])
    return summary, skills_formatted

# Advanced CSS for modern UI
custom_css = """
body {
    background: linear-gradient(to right, #fdfbe7, #e8f5e9);
    font-family: 'Segoe UI', sans-serif;
}
.gradio-container {
    padding: 20px;
    background-color: #ffffffd0;
    border-radius: 20px;
    box-shadow: 0 8px 24px rgba(0,0,0,0.1);
    margin: 30px auto;
    max-width: 900px;
}
.gr-button {
    background: linear-gradient(to right, #43cea2, #185a9d) !important;
    color: white !important;
    font-weight: bold !important;
    font-size: 16px !important;
    padding: 10px 20px !important;
    border-radius: 12px !important;
    border: none !important;
}
textarea, input, .gr-textbox {
    border-radius: 10px !important;
    font-size: 16px !important;
    padding: 10px !important;
}
h1, h2 {
    color: #2e7d32 !important;
    font-weight: 700 !important;
}
footer {
    display: none !important;
}
"""

# Gradio interface
with gr.Blocks(title="Resume Analyser") as demo:
    gr.Markdown("# Resume Summarizer & Skill Extractor")
    gr.Markdown("Upload your resume to instantly get a structured summary and highlight key skills.")

    with gr.Row():
        resume_file = gr.File(label="Upload Resume", file_types=[".docx", ".pdf"])
        submit_btn = gr.Button("Analyze")

    with gr.Row():
        summary_out = gr.Textbox(label="Resume Summary", lines=6, interactive=False)
        skills_out = gr.Textbox(label="Top Skills Extracted", lines=6, interactive=False)

    submit_btn.click(fn=process_resume, inputs=resume_file, outputs=[summary_out, skills_out])

# Setting debug=False prevents the cell from running indefinitely
demo.launch(debug=False, css=custom_css)